# Model Development for Ames Housing

This notebook covers:
1. Training multiple models
2. Hyperparameter tuning
3. Model comparison
4. Saving the best model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import json

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [2]:
# Load processed data
df = pd.read_csv('../data/processed/ames_processed.csv')
print(f'Dataset shape: {df.shape}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/ames_processed.csv'

In [ ]:
# Prepare features and target
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## Define Models

In [ ]:
def get_models():
    """Return dictionary of models to evaluate."""
    return {
        'Linear Regression': LinearRegression(),
        'Ridge': Ridge(random_state=42),
        'Lasso': Lasso(random_state=42),
        'ElasticNet': ElasticNet(random_state=42),
        'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(random_state=42),
        'XGBoost': XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
        'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
        'CatBoost': CatBoostRegressor(random_state=42, verbose=0),
        'SVR': SVR()
    }

## Train and Evaluate Models

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    """Train and evaluate a single model."""
    model.fit(X_train, y_train)
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, 
                                scoring='neg_mean_squared_error')
    cv_rmse = np.sqrt(-cv_scores)
    
    metrics = {
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'train_mae': mean_absolute_error(y_train, y_pred_train),
        'test_mae': mean_absolute_error(y_test, y_pred_test),
        'train_r2': r2_score(y_train, y_pred_train),
        'test_r2': r2_score(y_test, y_pred_test),
        'cv_rmse_mean': cv_rmse.mean(),
        'cv_rmse_std': cv_rmse.std()
    }
    
    return metrics, model


# Train all models
models = get_models()
results = []
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    try:
        metrics, trained_model = evaluate_model(model, X_train, y_train, X_test, y_test)
        metrics['model'] = name
        results.append(metrics)
        trained_models[name] = trained_model
        print(f'  Test RMSE: {metrics["test_rmse"]:.2f}, R²: {metrics["test_r2"]:.4f}')
    except Exception as e:
        print(f'  Error: {e}')

# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('test_rmse')
results_df

## Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# RMSE comparison
axes[0].barh(results_df['model'], results_df['test_rmse'])
axes[0].set_title('Model RMSE Comparison')
axes[0].set_xlabel('RMSE')
axes[0].axvline(results_df['test_rmse'].min(), color='green', linestyle='--', 
                label=f'Best: {results_df["test_rmse"].min():.2f}')
axes[0].legend()

# R² comparison
axes[1].barh(results_df['model'], results_df['test_r2'])
axes[1].set_title('Model R² Comparison')
axes[1].set_xlabel('R² Score')
axes[1].axvline(results_df['test_r2'].max(), color='green', linestyle='--', 
                label=f'Best: {results_df["test_r2"].max():.4f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Hyperparameter Tuning - Best Model

In [ ]:
# Get best model type
best_model_name = results_df.iloc[0]['model']
print(f'Best model: {best_model_name}')

# Define parameter grids for top models
param_grids = {
    'XGBoost': {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 0.9, 1.0],
        'colsample_bytree': [0.8, 0.9, 1.0]
    },
    'LightGBM': {
        'n_estimators': [100, 200, 300],
        'num_leaves': [31, 50, 80],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [5, 10, 15]
    },
    'CatBoost': {
        'iterations': [100, 200, 300],
        'depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'l2_leaf_reg': [1, 3, 5]
    },
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
}

# If best model is in param_grids, tune it
if best_model_name in param_grids:
    base_model = models[best_model_name]
    param_grid = param_grids[best_model_name]
    
    print(f'\nTuning {best_model_name}...')
    grid_search = GridSearchCV(
        base_model, param_grid, cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1, verbose=1
    )
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    print(f'Best parameters: {grid_search.best_params_}')
    print(f'Best CV score: {-grid_search.best_score_:.4f}')
else:
    # Use the best model from initial training
    best_model = trained_models[best_model_name]

# Evaluate tuned model
y_pred = best_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
final_r2 = r2_score(y_test, y_pred)
print(f'\nFinal Model Performance:')
print(f'RMSE: {final_rmse:.2f}')
print(f'R² Score: {final_r2:.4f}')

## Save Best Model

In [ ]:
# Save model
joblib.dump(best_model, '../models/best_model.pkl')
print('Model saved to models/best_model.pkl')

# Save metrics
model_metrics = {
    'model_name': best_model_name,
    'test_rmse': float(final_rmse),
    'test_r2': float(final_r2),
    'train_size': len(X_train),
    'test_size': len(X_test),
    'n_features': X_train.shape[1],
    'best_params': grid_search.best_params_ if best_model_name in param_grids else {}
}

with open('../models/model_metrics.json', 'w') as f:
    json.dump(model_metrics, f, indent=2)
print('Metrics saved to models/model_metrics.json')